[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/67_flow_matching_solution.ipynb)

# 🟡 Solution: Flow Matching Loss

Reference solution for the conditional flow matching (CFM) training objective.

Given source samples $x_0$, target samples $x_1$, and interpolation time $t \in [0,1]$, the optimal transport path defines a constant velocity field:

$$v^*(x_t, t) = x_1 - x_0$$

The loss is the mean squared error between the model's predicted velocity and this target:

$$\mathcal{L} = \mathbb{E}\left[\|f_\theta(x_t, t) - (x_1 - x_0)\|^2\right]$$

In [ ]:
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def flow_matching_loss(model_output, x0, x1, t):
    target_velocity = x1 - x0
    diff = model_output - target_velocity
    return (diff * diff).mean()

In [ ]:
torch.manual_seed(0)

B, D = 16, 8
x0 = torch.randn(B, D)
x1 = torch.randn(B, D)
t  = torch.rand(B)

# Perfect prediction => loss should be exactly 0
perfect_output = x1 - x0
loss_perfect = flow_matching_loss(perfect_output, x0, x1, t)
print(f"Perfect prediction => loss = {loss_perfect.item():.6f}  (expected 0.0)")

# Random prediction => loss should be positive
random_output = torch.randn(B, D)
loss_random = flow_matching_loss(random_output, x0, x1, t)
print(f"Random prediction  => loss = {loss_random.item():.4f}   (expected > 0)")

# Slightly off prediction => loss between 0 and random
noisy_output = perfect_output + 0.1 * torch.randn(B, D)
loss_noisy = flow_matching_loss(noisy_output, x0, x1, t)
print(f"Noisy prediction   => loss = {loss_noisy.item():.4f}   (expected small but > 0)")

In [ ]:
from torch_judge import check
check("flow_matching")